# 00 — Setup & API sanity check

Project: **NBA Unicorn & Breakout Hunting**. Could we have identified future NBA stars/unicorns *before* they became obvious, using only information available at the time?

This notebook: environment check (Step 0) and a first look at what `nba_api` returns for a single player (Step 1A).

Leakage rule from day one: any feature for season *t* may use only data from seasons ≤ *t*.

## Step 0 — Environment check

In [1]:
import sys
import pandas as pd
import numpy as np
import nba_api

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("nba_api:", nba_api.__version__)

Python: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 21:00:12) [Clang 16.0.6 ]
pandas: 3.0.6
numpy: 2.5.3
nba_api: 1.11.4


In [2]:
from nba_api.stats.static import players

shai = players.find_players_by_full_name(
    "Shai Gilgeous-Alexander"
)

shai

[{'id': 1628983,
  'full_name': 'Shai Gilgeous-Alexander',
  'first_name': 'Shai',
  'last_name': 'Gilgeous-Alexander',
  'is_active': True}]

In [3]:
from nba_api.stats.endpoints import playercareerstats

shai_id = shai[0]["id"]

career = playercareerstats.PlayerCareerStats(
    player_id=shai_id
)

shai_stats = career.season_totals_regular_season.get_data_frame()

shai_stats

,PLAYER_ID,SEASON_ID,LEAGUE_ID,TEAM_ID,TEAM_ABBREVIATION,PLAYER_AGE,GP,GS,MIN,FGM,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS
0,1628983,2018-19,00,1610612746,LAC,20.0,82,73,2174,341,...,0.800,57,175,232,270,96,45,141,175,889
1,1628983,2019-20,00,1610612760,OKC,21.0,70,70,2428,478,...,0.807,47,365,412,232,79,47,134,117,1331
2,1628983,2020-21,00,1610612760,OKC,22.0,35,35,1180,287,...,0.808,19,147,166,207,27,23,106,70,830
3,1628983,2021-22,00,1610612760,OKC,23.0,56,56,1942,477,...,0.810,37,241,278,331,71,46,154,142,1371
4,1628983,2022-23,00,1610612760,OKC,24.0,68,68,2416,704,...,0.905,59,270,329,371,112,65,192,192,2135
5,1628983,2023-24,00,1610612760,OKC,25.0,75,75,2553,796,...,0.874,65,350,415,465,150,67,162,184,2254
6,1628983,2024-25,00,1610612760,OKC,26.0,76,76,2598,860,...,0.898,67,312,379,486,131,77,183,164,2484
7,1628983,2025-26,00,1610612760,OKC,27.0,68,68,2259,731,...,0.879,38,254,292,448,95,52,151,139,2117


## Step 1A — What does `PlayerCareerStats` return?

One player's career endpoint. We use it to understand the raw material, **not** to build the main dataset (that would mean ~1,500 calls, and much of what it returns is career aggregates that leak future information).

⚠️ `career.expected_data` is ordered alphabetically, but the API returns datasets in a different order. Zipping the two mislabels every table. Always read names from the response itself, or use the named attributes.

In [4]:
for rs in career.get_dict()["resultSets"]:
    print(f"{rs['name']:32s} rows={len(rs['rowSet']):3d} cols={len(rs['headers'])}")

SeasonTotalsRegularSeason        rows=  8 cols=27
CareerTotalsRegularSeason        rows=  1 cols=24
SeasonTotalsPostSeason           rows=  5 cols=27
CareerTotalsPostSeason           rows=  1 cols=24
SeasonTotalsAllStarSeason        rows=  3 cols=27
CareerTotalsAllStarSeason        rows=  1 cols=24
SeasonTotalsCollegeSeason        rows=  1 cols=27
CareerTotalsCollegeSeason        rows=  1 cols=24
SeasonTotalsShowcaseSeason       rows=  0 cols=27
CareerTotalsShowcaseSeason       rows=  0 cols=24
SeasonRankingsRegularSeason      rows=  8 cols=27
SeasonRankingsPostSeason         rows=  5 cols=27
SeasonHighs                      rows= 21 cols=11
CareerHighs                      rows= 19 cols=11


| Dataset | Use for this project |
|---|---|
| `SeasonTotalsRegularSeason` | Player × season × team box-score totals. Traded players get one row per team **plus** a `TOT` row (`TEAM_ID = 0`). |
| `CareerTotals*` | ⛔ Career sums include future seasons, so they are a leakage hazard. Never use them as features. |
| `SeasonTotalsPostSeason` | Playoffs. Possibly a later feature or outcome. |
| `SeasonTotalsAllStarSeason` | ⛔ All-Star selections are an *outcome*, never a feature. |
| `SeasonTotalsCollegeSeason` | Pre-NBA production. A useful draft-time prior later. |
| `SeasonRankings*` | League rank for each stat in each season. Redundant with raw stats. |
| `SeasonHighs` / `CareerHighs` | Single-game highs. Noisy, so ignore. |

In [5]:
print(shai_stats.columns.tolist())
shai_stats[["SEASON_ID", "TEAM_ABBREVIATION", "PLAYER_AGE", "GP", "GS", "MIN", "PTS"]]

['PLAYER_ID', 'SEASON_ID', 'LEAGUE_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'PLAYER_AGE', 'GP', 'GS', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']


,SEASON_ID,TEAM_ABBREVIATION,PLAYER_AGE,GP,GS,MIN,PTS
0,2018-19,LAC,20.0,82,73,2174,889
1,2019-20,OKC,21.0,70,70,2428,1331
2,2020-21,OKC,22.0,35,35,1180,830
3,2021-22,OKC,23.0,56,56,1942,1371
4,2022-23,OKC,24.0,68,68,2416,2135
5,2023-24,OKC,25.0,75,75,2553,2254
6,2024-25,OKC,26.0,76,76,2598,2484
7,2025-26,OKC,27.0,68,68,2259,2117


Observations: box-score totals only (no usage, TS%, ratings, height or position). `PLAYER_AGE` is an integer age at a reference date the NBA chooses; we'll compute age ourselves from birth date later.

### Per-dataset inspection: shape, columns, first rows, dtypes

`career.get_data_frames()` returns frames in **response order**, so we pair them with the response's own names. The assert checks that the pairing is correct.

In [6]:
names = [rs["name"] for rs in career.get_dict()["resultSets"]]
career_dfs = dict(zip(names, career.get_data_frames()))
assert career_dfs["SeasonTotalsRegularSeason"].equals(shai_stats)

pd.DataFrame({
    name: {"rows": df.shape[0], "cols": df.shape[1],
           "numeric_cols": df.select_dtypes("number").shape[1],
           "first_cols": ", ".join(df.columns[:6])}
    for name, df in career_dfs.items()
}).T

,rows,cols,numeric_cols,first_cols
SeasonTotalsRegularSeason,8,27,24,"PLAYER_ID, SEASON_ID, LEAGUE_ID, TEAM_ID, TEAM..."
CareerTotalsRegularSeason,1,24,23,"PLAYER_ID, LEAGUE_ID, TEAM_ID, GP, GS, MIN"
SeasonTotalsPostSeason,5,27,24,"PLAYER_ID, SEASON_ID, LEAGUE_ID, TEAM_ID, TEAM..."
CareerTotalsPostSeason,1,24,23,"PLAYER_ID, LEAGUE_ID, TEAM_ID, GP, GS, MIN"
SeasonTotalsAllStarSeason,3,27,24,"PLAYER_ID, SEASON_ID, LEAGUE_ID, TEAM_ID, TEAM..."
CareerTotalsAllStarSeason,1,24,23,"PLAYER_ID, LEAGUE_ID, TEAM_ID, GP, GS, MIN"
SeasonTotalsCollegeSeason,1,27,24,"PLAYER_ID, SEASON_ID, LEAGUE_ID, ORGANIZATION_..."
CareerTotalsCollegeSeason,1,24,23,"PLAYER_ID, LEAGUE_ID, ORGANIZATION_ID, GP, GS,..."
SeasonTotalsShowcaseSeason,0,27,0,"PLAYER_ID, SEASON_ID, LEAGUE_ID, TEAM_ID, TEAM..."
CareerTotalsShowcaseSeason,0,24,0,"PLAYER_ID, LEAGUE_ID, TEAM_ID, GP, GS, MIN"


In [7]:
# The table we actually care about: dtypes
shai_stats.dtypes.value_counts()

int64      20
float64     4
str         3
Name: count, dtype: int64

In [8]:
# Non-numeric columns: these are the IDs/keys, stored as strings
shai_stats.select_dtypes(exclude="number").head(3)

,SEASON_ID,LEAGUE_ID,TEAM_ABBREVIATION
0,2018-19,00,LAC
1,2019-20,00,OKC
2,2020-21,00,OKC


In [9]:
# Pre-NBA data that is available at draft time, so allowed for early-career features
career_dfs["SeasonTotalsCollegeSeason"]

,PLAYER_ID,SEASON_ID,LEAGUE_ID,ORGANIZATION_ID,SCHOOL_NAME,PLAYER_AGE,GP,GS,MIN,FGM,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS
0,1628983,2017-18,00,2852,Kentucky,19.0,37,24,1248,183,...,0.822,33,120,153,189,61,18,99,62,532


In [10]:
# Single-game highs: different schema; noisy, likely unused
career_dfs["SeasonHighs"].head(3)

,PLAYER_ID,GAME_ID,GAME_DATE,VS_TEAM_ID,VS_TEAM_CITY,VS_TEAM_NAME,VS_TEAM_ABBREVIATION,STAT,STAT_VALUE,STAT_ORDER,DATE_EST
0,1628983,0042500143,Apr 25 2026,1610612756,Phoenix,Suns,PHX,PTS,42,1,2026-04-25T00:00:00
1,1628983,0042500142,Apr 22 2026,1610612756,Phoenix,Suns,PHX,REB,5,2,2026-04-22T00:00:00
2,1628983,0042500223,May 09 2026,1610612747,Los Angeles,Lakers,LAL,OREB,2,3,2026-05-09T00:00:00
